# 35 · Sample satellite files at points against h5py and pyproj

**Question.** When Fieldwork's Sample satellite files at points reads a Sentinel-5P Level 2 file, does each point get the pixel whose centre is nearest it under Fieldwork's declared rule, with the value, quality and time an independent HDF5 reader finds there? And does each row say the gap and quality outcome the declared rule gives, under both quality operators?

Fieldwork experiment 179. The NYC worked example reads the four NO₂ monitors this way. The file here is synthetic, shaped like the offline NO₂ file: no Copernicus data.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["numpy", "h5py", "pyproj"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


## Parity

h5py reads the whole swath at once, and the pixel is recomputed by comparing every centre, so a point Fieldwork's chunk-by-chunk search places wrongly, or stops looking for too early, disagrees. The distance is Fieldwork's declared approximation, degrees with the longitude scaled by the cosine of the latitude; the geodesically nearest centre from pyproj is reported beside it as a measurement, not part of the criterion.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_35_sample_satellite import run

report = run(Path.cwd() / "fixtures" / "sample-satellite-s5p-001.json")
print("agrees:", report["agrees"])
for x in report["placement"]:
    print(f"{x['point']:<16} fieldwork {x['fieldwork']}  declared rule {x['declaredRule']}  "
          f"{'same' if x['samePixel'] and x['valueQualityTimeAgree'] else 'DIFFERS'}")
g = report["geodesic"]
print(f"geodesic nearest = declared rule at {g['same']} of {g['of']} placed points")
for c in report["cases"]:
    print(f"qa_value {c['operator']} 0.75: {c['rows']['fieldwork']} rows; {c['counts']['fieldwork']}; "
          f"{'agrees' if c['agrees'] else 'differs at ' + str(c['rowsDiffering'])}")


## What this cannot establish

Anything about a real orbit file's content: the file is synthetic. Nor reading a remote file by HTTP range, which is where the cost is, nor whether the nearest centre is the right pixel for a study. h5py and Fieldwork's h5wasm both sit on libhdf5, so a defect in libhdf5 itself would be shared.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture:", report["fixture"]["id"], report["fixture"]["operation"]["widget"], report["fixture"]["operation"]["release"])
